In [1]:
import json
import time
import joblib
import pandas as pd
import numpy as np

from datetime import datetime
from pathlib import Path
from kafka import KafkaProducer

print("Producer libraries imported successfully!")

Producer libraries imported successfully!


In [2]:
TEST_PATH = Path("test_fraud.csv")

FEATURES_PATH = Path(
    "final_model_feature_columns.pkl"
)

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Test file not found: {TEST_PATH}"
    )

if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Feature file not found: {FEATURES_PATH}"
    )

stream_data = pd.read_csv(TEST_PATH)
feature_columns = joblib.load(FEATURES_PATH)

stream_data.columns = (
    stream_data.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("Test transactions:", stream_data.shape)
print("Model features:", len(feature_columns))

Test transactions: (8651, 45)
Model features: 41


In [3]:
missing_features = [
    column
    for column in feature_columns
    if column not in stream_data.columns
]

if missing_features:

    print(
        "Missing features will be sent as null:"
    )

    print(missing_features)

    for column in missing_features:
        stream_data[column] = np.nan

else:
    print("All model features are available!")

All model features are available!


In [4]:
KAFKA_SERVER = "localhost:9092"
TRANSACTION_TOPIC = "fraud-transactions"

producer = KafkaProducer(
    bootstrap_servers=KAFKA_SERVER,
    value_serializer=lambda value: json.dumps(
        value,
        default=str
    ).encode("utf-8"),
    acks="all",
    retries=3
)

print("Producer connected successfully!")

Producer connected successfully!


In [5]:
def make_json_safe(value):

    if pd.isna(value):
        return None

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    return value

In [6]:
NUMBER_OF_TRANSACTIONS = 100
DELAY_SECONDS = 1

stream_sample = (
    stream_data
    .sample(
        n=min(
            NUMBER_OF_TRANSACTIONS,
            len(stream_data)
        ),
        random_state=42
    )
    .reset_index(drop=True)
)

transactions_to_send = len(
    stream_sample
)

In [7]:
transactions_to_send = len(
    stream_sample
)

print("Starting Kafka producer...\n")

for position in range(
    transactions_to_send
):

    transaction_row = (
        stream_sample.iloc[position]
    )

    transaction_message = {
        column: make_json_safe(
            transaction_row[column]
        )
        for column in feature_columns
    }

    transaction_message["stream_id"] = (
        "TXN-"
        + datetime.now().strftime(
            "%Y%m%d%H%M%S%f"
        )
        + f"-{position + 1}"
    )

    transaction_message["produced_at"] = (
        datetime.now().isoformat()
    )

    future = producer.send(
        TRANSACTION_TOPIC,
        value=transaction_message
    )

    metadata = future.get(
        timeout=10
    )

    print(
        f"Transaction {position + 1} sent | "
        f"Partition: {metadata.partition} | "
        f"Offset: {metadata.offset}"
    )

    if DELAY_SECONDS > 0:
        time.sleep(DELAY_SECONDS)

producer.flush()
producer.close()

print(
    "\nAll transactions sent successfully!"
)

Starting Kafka producer...

Transaction 1 sent | Partition: 0 | Offset: 500
Transaction 2 sent | Partition: 0 | Offset: 501
Transaction 3 sent | Partition: 0 | Offset: 502
Transaction 4 sent | Partition: 0 | Offset: 503
Transaction 5 sent | Partition: 0 | Offset: 504
Transaction 6 sent | Partition: 0 | Offset: 505
Transaction 7 sent | Partition: 0 | Offset: 506
Transaction 8 sent | Partition: 0 | Offset: 507
Transaction 9 sent | Partition: 0 | Offset: 508
Transaction 10 sent | Partition: 0 | Offset: 509
Transaction 11 sent | Partition: 0 | Offset: 510
Transaction 12 sent | Partition: 0 | Offset: 511
Transaction 13 sent | Partition: 0 | Offset: 512
Transaction 14 sent | Partition: 0 | Offset: 513
Transaction 15 sent | Partition: 0 | Offset: 514
Transaction 16 sent | Partition: 0 | Offset: 515
Transaction 17 sent | Partition: 0 | Offset: 516
Transaction 18 sent | Partition: 0 | Offset: 517
Transaction 19 sent | Partition: 0 | Offset: 518
Transaction 20 sent | Partition: 0 | Offset: 519
T